# Soldani - Second task - Benchmark

## 1. Setup del path di progetto

Individua automaticamente la directory radice del progetto cercando la cartella src/ nella gerarchia superiore, quindi la aggiunge a sys.path per consentire gli import assoluti.


In [54]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Import librerie e configurazione client LLM

Importa pandas, json, pgmpy e i moduli causali di FairMind (build_sfm, fit_discrete_bayesian_model, effetti). L'endpoint del server llama.cpp viene letto dalle variabili d'ambiente LLAMA_HOST/LLAMA_PORT (fallback localhost:8080), cosi' lo stesso notebook funziona sia in locale sia su THOR.


In [ ]:
import json
import os
import pandas as pd

from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.llm import LLM_CONFIGS

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"
print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

## 3. Configurazione del benchmark

Definisce il dizionario CONFIG con il dataset Adult: attributo protetto (S2_gender, Female/Male), target (T_income, >50K), mediatore (hours-per-week), confounder (education).


In [56]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

## 4. FairMind — calcolo del ground truth

Costruisce lo Standard Fairness Model (SFM), fitta la Bayesian Network con stima BDeu e calcola TV, TE, DE, IE tramite inferenza causale formale. SE si ricava come `TV - TE` (Eq. 3, Plecko & Bareinboim 2024) — non da `spurious_effect()` (che calcola una quantità a un solo argomento, diversa dalla SE ufficiale a due argomenti). Questi valori costituiscono il ground truth per il confronto.


In [ ]:
import time

def run_fairmind(config: dict) -> tuple[dict, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    # Stesso binning usato in build_llm_prompt per hours-per-week: FairMind e LLM
    # devono vedere la stessa rappresentazione del mediatore, altrimenti DE/IE
    # non sono confrontabili (erano calcolati su raw vs binned).
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"],
            bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"],
            include_lowest=True,
        )

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    tv = total_variation(bn, target, config["protected"], x0, x1)
    te = total_effect(bn, target, config["protected"], x0, x1)
    effects = {
        "TV": tv,
        "TE": te,
        # SE = TV - TE (Eq. 3, Plecko & Bareinboim 2024) — NON spurious_effect()
        # da sola: quella calcola P(y|x)-P(y|do(x)) per un solo x, una quantita'
        # diversa dalla SE ufficiale a due argomenti usata nel resto del paper.
        "SE": tv - te,
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, elapsed

ground_truth, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind — elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

## 5. Costruzione del prompt per il LLM

Pre-aggrega le probabilità condizionali dal dataset in 5 tabelle CSV (P(Y|X), P(Z), P(Y|X,Z), P(W|X,Z), P(Y|X,W,Z)) e assembla il prompt testuale con le formule di identificazione per il LLM. Chiede solo TV, TE, DE, IE (non SE, ridondante: si ricava per sottrazione da TV e TE).


In [ ]:
def build_llm_prompt(config: dict) -> str:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    protected = config["protected"]
    target = config["target_col"]
    target_val = config["target_val"]
    confounders = config["confounders"]

    # Binning delle variabili continue nei mediators
    # hours-per-week -> fasce standard (part-time, full-time, overtime, ecc.)
    binned_mediators = []
    for col in config["mediators"]:
        if col == "hours-per-week":
            bin_col = f"{col}_bin"
            df[bin_col] = pd.cut(
                df[col],
                bins=[0, 20, 35, 45, 60, 100],
                labels=["<=20", "21-35", "36-45", "46-60", ">60"],
                include_lowest=True,
            )
            binned_mediators.append(bin_col)
        else:
            binned_mediators.append(col)

    mediators = binned_mediators

    # Y binario: 1 se target_val, altrimenti 0
    df["_y"] = (df[target] == target_val).astype(int)

    # --- 1. P(Y=y | X) ---
    p_y_given_x = (
        df.groupby(protected, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X)"})
    )

    # --- 2. P(Z) — distribuzione marginale dei confounders ---
    p_z = (
        df.groupby(confounders, observed=True).size().reset_index(name="count")
    )
    p_z["P(Z)"] = (p_z["count"] / len(df)).round(4)
    p_z = p_z.drop(columns="count")

    # --- 3. P(Y=y | X, Z) ---
    p_y_given_xz = (
        df.groupby([protected] + confounders, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X,Z)"})
    )

    # --- 4. P(W | X, Z) ---
    p_w_given_xz = (
        df.groupby([protected] + confounders + mediators, observed=True).size()
        .reset_index(name="count")
    )
    group_totals = p_w_given_xz.groupby([protected] + confounders, observed=True)["count"].transform("sum")
    p_w_given_xz["P(W|X,Z)"] = (p_w_given_xz["count"] / group_totals).round(4)
    p_w_given_xz = p_w_given_xz.drop(columns="count")

    # --- 5. P(Y=y | X, W, Z) ---
    p_y_given_xwz = (
        df.groupby([protected] + mediators + confounders, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X,W,Z)"})
    )

    def to_compact_csv(d: pd.DataFrame) -> str:
        return d.to_csv(index=False)

    return f"""You are a causal fairness expert. Compute four causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

You are given PRE-AGGREGATED CONDITIONAL PROBABILITY TABLES computed from the dataset
(n={len(df)} rows). Use these tables directly — do not assume access to raw data.
Note: "hours-per-week" has been discretized into bins: <=20, 21-35, 36-45, 46-60, >60.

VARIABLE ROLES:
- X (protected): "{protected}", x0="{config['x0']}", x1="{config['x1']}"
- Y (target):    "{target}", target state="{target_val}"
- W (mediators): {mediators}
- Z (confounders): {confounders}

TABLE 1 — P(Y=y | X):
{to_compact_csv(p_y_given_x)}

TABLE 2 — P(Z):
{to_compact_csv(p_z)}

TABLE 3 — P(Y=y | X, Z):
{to_compact_csv(p_y_given_xz)}

TABLE 4 — P(W | X, Z):
{to_compact_csv(p_w_given_xz)}

TABLE 5 — P(Y=y | X, W, Z):
{to_compact_csv(p_y_given_xwz)}

IDENTIFICATION FORMULAE (use these exactly, aggregating over TABLE rows as needed):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)                    [from TABLE 1]
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)          [from TABLE 3, TABLE 2]
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)   [from TABLE 5, TABLE 4, TABLE 2]
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)       [from TABLE 5, TABLE 4, TABLE 2]

Note: the Spurious Effect (SE) is NOT requested here — it is fully determined
by SE = TV - TE (Plecko & Bareinboim, 2024), so it is derived afterwards
from your TV and TE values rather than computed independently.

INSTRUCTIONS:
For DE and IE, the sums run over EVERY combination of z (each row of TABLE 2) and
w (each bin of hours-per-week). For each (z,w) pair you MUST look up the matching
row in TABLE 4 and TABLE 5 by z and w together — do not skip or approximate any term.

Show your work as a short step-by-step calculation for DE and IE (one line per (z,w)
term is fine, or grouped by z), THEN give the final answer.

End your response with a line "FINAL_JSON:" followed by ONLY the JSON object below,
with no markdown formatting:
{{
  "TV": <float>,
  "TE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_llm_prompt(CONFIG)
print(prompt[:2000], "\n[...]")
print(f"\nTotal prompt length (chars): {len(prompt)}")

## 6. Chiamata al LLM (Qwen2.5-7B) e parsing della risposta

Invia il prompt con le tabelle pre-aggregate al modello Qwen2.5-7B via llama.cpp, raccoglie le metriche di tempo e token, estrae il JSON (TV, TE, DE, IE) dalla risposta tramite regex e lo parsifica. SE viene poi calcolata come `TV - TE` sui valori restituiti dall'LLM.


In [ ]:
from src.llm import call_llm

llm_effects, llm_usage, llm_time = call_llm(prompt)

# SE non viene chiesta all'LLM (v. nota nel prompt): si ricava qui con la
# stessa identita' usata per il ground truth (SE = TV - TE), cosi' il
# confronto sulla SE riflette solo gli errori di TV/TE dell'LLM, non un
# calcolo extra e ridondante.
llm_effects["SE"] = llm_effects["TV"] - llm_effects["TE"]

print(f"LLM — time: {llm_time:.4f}s")
print(f"Token: input={llm_usage['input_tokens']}, "
      f"output={llm_usage['output_tokens']}, "
      f"total={llm_usage['total_tokens']}")
print(json.dumps(llm_effects, indent=2))

## 7. Confronto FairMind vs LLM — tabella discrepancies

Calcola l'errore assoluto e relativo percentuale tra il ground truth (FairMind) e la predizione del LLM per ognuno dei 5 effetti causali, producendo una tabella riassuntiva.


In [60]:
def compute_discrepancies(ground_truth: dict, llm_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        llm_val = float(llm_effects.get(effect, float("nan")))
        abs_err = abs(gt - llm_val)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
        "fairmind":    round(gt,  6),
        "llm":         round(llm_val, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, llm_effects)
print(discrepancies.to_string(index=False))

effect  fairmind       gpt  abs_error  rel_error_%
    TV  0.194470  0.194500   0.000030         0.02
    TE  0.183161  0.231167   0.048005        26.21
    SE -0.007296 -0.036667   0.029371       402.56
    DE  0.137049  0.083167   0.053883        39.32
    IE -0.046112 -0.047000   0.000888         1.93


## 8. Salvataggio dei risultati su file JSON

Salva l'intero risultato del benchmark in un file JSON dentro benchmark_results/, includendo configurazione, effetti FairMind, effetti LLM, discrepancies, metriche token e timing.


In [61]:
def save_results(config, ground_truth, llm_effects, discrepancies, usage, fairmind_time, llm_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "llm":           llm_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "llm_seconds":      round(llm_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, llm_effects, discrepancies, llm_usage, fairmind_time, llm_time)

Saved: benchmark_results/adult_20260712_161949.json
